# 16 · Protocol flow (Left -> S3 -> Right) and packaging

This notebook runs the distributed protocol by hand on one real pair: the Left side counts and publishes to a real S3 scratch prefix, the Right side (here: the SQL warehouse through `WarehouseTarget`, standing in for the Databricks job's `SparkTarget`) reconciles and writes manifests, the Left side pulls windows and commits units, the Right side compares them, and the report is built, published and fetched back.
It mirrors `tests/unit/test_protocol_flow.py` (same calls, real connections instead of CSV fixtures) and `tests/unit/test_packaging.py` (read-only: dependency boundaries, what the Right job bundles, and a `DRY_RUN` Workspace upload plan).
The only writes are the S3 scratch prefix (deleted at the end unless `KEEP=True`) and a local work folder.

## How to approach

- **Prerequisites**: kernel = the repo `.venv` (`uv sync --extra dbx`, plus `--extra athena` / `--extra ldap` if the pair needs them). The repo `.env` holds the Left credentials. AWS credentials for the scratch bucket come from the default boto3 chain (`AWS_PROFILE`, or keep `dtrack dbx s3 creds --watch` running); `DTRACK_S3_ENDPOINT_URL` is honoured. `~/.databrickscfg` has `DBX_PROFILE`; `DTRACK_DBX_WAREHOUSE_ID` names a running SQL warehouse.
- **Run order**: strictly top to bottom; each protocol step reads what the previous one wrote to S3. Steps 1-2 are quick (counts). Step 3 pulls every matching window of the window range from the Left (guarded by `MAX_ROWS`); step 4 pulls the same windows from the warehouse. Section 6 (`dtrack_local.run_all`) repeats the whole flow in one call and is off unless `RUN_LOCAL_E2E = True`. Section 7 (packaging) needs no connections except the Workspace dry-run.
- **What to look at**: the S3 keys after every step (`row_check/`, `manifest/`, `_INDEX/`, `units/`, `results/`, `dead/`), the row-check summary and gate, the per-unit verdicts, `summary.json` status `DONE` with no `dead/` units, and the packaging boundary scans (all empty).
- **Failure modes**: `AccessDenied` on the scratch prefix = AWS profile / bucket policy; Left `preflight failed` = credentials or a missing extra; `row check has not passed` (no manifest) = no day with equal counts, widen `DAYS`; units in `dead/` = a Right pull failed (read the error in the `dead/*.json`); `left heartbeat is stale` should not happen here because the notebook raises the timeout.

In [ ]:
import os
from pathlib import Path

CONFIG_PATH = os.environ.get("DTRACK_PROD_CONFIG", "C:/work/dtrack/c.json")      # your real pairs config
PAIR = os.environ.get("DTRACK_NB_PAIR", "orders")                               # one active pair in it
TO_DATE = None                                                                    # None = yesterday; or "YYYY-MM-DD"
DAYS = 3                                                                          # window = DAYS days ending at TO_DATE
S3_SCRATCH = os.environ.get("DTRACK_PROD_S3", "s3://<bucket>/dtrack/nb-scratch")  # scratch prefix; this run uses a sub-folder
DBX_PROFILE = os.environ.get("DTRACK_DBX_PROFILE", "corp")                        # ~/.databrickscfg profile
WAREHOUSE_ID = os.environ.get("DTRACK_DBX_WAREHOUSE_ID", "")                      # SQL warehouse id for the Right side
DISPOSITION = os.environ.get("DTRACK_LOCAL_DISPOSITION", "external_links")        # "inline" when presigned downloads are blocked
ENV_FILE = os.environ.get("DTRACK_ENV_FILE")                                      # None = <repo>/.env
GATE = "auto"                                                                     # row-check gate: auto | manual | ratio:0.1
MAX_ROWS = 500_000                                                                # refuse the Stage 2 pull above this many Left rows
RUN_LOCAL_E2E = False                                                             # section 6: the same flow via dtrack_local.run_all
BUILD_WHEEL = False                                                               # section 7: `uv build --wheel` into WORKDIR (local only)
WORKSPACE_TARGET = None                                                           # None = /Users/<me>/dtrack_right
OVERWRITE = False                                                                 # only used when DRY_RUN = False
WORKDIR = Path.home() / ".local/plans/nbs/_work/16"                               # local scratch folder
KEEP = False                                                                      # keep the S3 scratch prefix and WORKDIR
DRY_RUN = True                                                                    # True = never write to the Databricks Workspace

In [ ]:
# Imports used across the notebook
import json
import logging
import shutil
from datetime import date, timedelta
from types import SimpleNamespace

import pandas as pd
from dotenv import load_dotenv

import dtrack_left
from dtrack_left.backends.databricks import workspace
from dtrack_left.backends.sources import source_from_config
from dtrack_left.cli.config import load_config
from dtrack_local.pipeline import WarehouseTarget
from dtrack_local.warehouse import Warehouse

# The repo root (editable install: <repo>/src/dtrack_left/__init__.py) and its .env with the
# Left credentials (<macro>_USR / _PWD / _DSN, ORACLE_*, LDAP_DSN, DTRACK_ORACLE_MACROS ...)
REPO = Path(dtrack_left.__file__).resolve().parents[2]
env_file = Path(ENV_FILE) if ENV_FILE else REPO / ".env"
loaded = load_dotenv(env_file, override=False)
print("repo :", REPO)
print(".env :", env_file, "(loaded)" if loaded else "(not found - credentials must already be in the environment)")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)

# The real pairs config, validated exactly like `dtrack row-check` does
config = load_config(CONFIG_PATH)
active = [name for name, spec in config["pairs"].items() if not spec.get("skip")]
print("active pairs:", active)
if PAIR not in active:
    raise KeyError(f"PAIR={PAIR!r} is not an active pair in {CONFIG_PATH}")

# The pair, deep-copied, with its date range narrowed to the check window (TO_DATE back DAYS days)
to_date = TO_DATE or (date.today() - timedelta(days=1)).isoformat()
from_date = (date.fromisoformat(to_date) - timedelta(days=DAYS - 1)).isoformat()
pair = json.loads(json.dumps(config["pairs"][PAIR]))
pair["fromDate"], pair["toDate"] = from_date, to_date
print(f"pair {PAIR}: window {from_date} .. {to_date}")
print("  left :", {k: v for k, v in pair["left"].items() if k not in ("user", "password", "dsn")})
print("  right:", pair["right"])

WORKDIR.mkdir(parents=True, exist_ok=True)

# The two store implementations: the Left package and the Right job each have their own S3Store
from dtrack_left.backends.store import S3Store as LeftS3Store, fetch_report
from dtrack_left.checks.col_check import ColCheckPipeline, LeftRuntime, publish_fixed_keys, wait_for_manifests
from dtrack_left.checks.row_check import make_run_id, public_pair_config, run_row_check
from dtrack_right.backends.store import open_store, resolve_run
from dtrack_right.checks.col_check import ColCheck
from dtrack_right.checks.row_check import run_row_check as run_right_row_check
from dtrack_right.report import build_report, publish_report

# A one-pair config for this run, and a run id derived from it (as `dtrack row-check` does)
run_config = {"pairs": {PAIR: pair}}
RUN_ID = make_run_id(run_config)
SCRATCH = f"{S3_SCRATCH.rstrip('/')}/nb16-{RUN_ID}"
WORK_ROOT = f"{SCRATCH}/runs"
REPORT_URI = f"{SCRATCH}/report"
print("run id   :", RUN_ID)
print("work root:", WORK_ROOT)
print("report   :", REPORT_URI)

# Helper: list what is under the run folder right now, grouped by top-level prefix
def show_keys(store, title):
    keys = store.list_keys("")
    groups = {}
    for key in keys:
        groups.setdefault(key.split("/")[0], []).append(key)
    print(f"--- {title}: {len(keys)} object(s)")
    for group, items in sorted(groups.items()):
        print(f"  {group:12} {len(items):4}  e.g. {items[:3]}")

## 0 · Scratch prefix round trip

Both store classes must read what the other writes at the same URI: write with the Left `S3Store`, read and list with the Right one.

In [ ]:
# Write with the Left class, read back with the Right class (the whole protocol relies on this)
probe_left = LeftS3Store(f"{SCRATCH}/s3-check")
probe_right = open_store(f"{SCRATCH}/s3-check")
probe_left.put_json("probe.json", {"ok": True, "run_id": RUN_ID})
print("right reads :", probe_right.get_json("probe.json"))
print("right lists :", probe_right.list_keys(""))
print("exists      :", probe_right.exists("probe.json"), "| missing key exists:", probe_right.exists("nope.json"))

## Connect

The real Left source for preflight, and the warehouse that plays the Right side. `target_factory` is what the Right job receives: one target per pair's Right config.

In [ ]:
# The real Left source (Oracle / Athena / file / duckdb) and the real Right side through the SQL warehouse
source = source_from_config(pair["left"])
warehouse = Warehouse(workspace(DBX_PROFILE), WAREHOUSE_ID, disposition=DISPOSITION)
right = WarehouseTarget(warehouse, pair["right"], WORKDIR / "right")

# Preflight both sides before any counting: Left SELECT ... WHERE 1=0, Right DESCRIBE QUERY
left_health = source.preflight()
right_health = right.preflight()
print("left  preflight:", left_health)
print("right preflight:", {k: v for k, v in right_health.items() if k != "columns"})
if not (left_health["ok"] and right_health["ok"]):
    raise RuntimeError("fix the preflight error above before going on")

# What the Right side calls for every pair (the Databricks job passes SparkTarget(spark, config) instead)
def target_factory(right_config):
    return WarehouseTarget(warehouse, right_config, WORKDIR / "right")

## 1 · Left: row check (counts -> S3)

`run_row_check` preflights, counts each day and writes `row_check/<pair>.left.json` with a credential-free copy of the pair config, then `row_check/_LEFT_CLOSED`, a heartbeat and `_STATUS.json`.

In [ ]:
# Left side, Stage 1: the same call as `dtrack row-check`
left_store = LeftS3Store(f"{WORK_ROOT}/{RUN_ID}")
run_id, documents = run_row_check(run_config, left_store, run_id=RUN_ID)
left_doc = documents[0]
print("counts:", left_doc["counts"])
print("bounds:", left_doc["bounds"], "| left count took", left_doc["timing"]["left_count_s"], "s")
show_keys(left_store, "after the Left row check")

# What crosses to the Right side: no user / password / dsn / *_env fields
print("published left config:", left_doc["left"])
print("secret fields stripped:", sorted(set(pair["left"]) - set(public_pair_config(pair)["left"])) or "(none were set in the config)")

## 2 · Right: reconcile, gate, manifest

The Right job finds the run under the work root (`resolve_run`), counts its own side, reconciles, gates, writes `reports/<pair>_compare_row.csv`, `manifest/<pair>.json` when the gate passes, the verdict `row_check/<pair>.json` last, and finally `row_check/_CLOSED`.

In [ ]:
# Right side finds the newest unfinished run under the work root, exactly as dtrack_right.job does
right_store, resolved = resolve_run(WORK_ROOT)
print("resolved run:", resolved, "| same as ours:", resolved == RUN_ID)

processed = run_right_row_check(right_store, target_factory, gate=GATE)
verdict = right_store.get_json(f"row_check/{PAIR}.json")
print("pairs processed:", processed)
print("summary:", verdict["summary"])
print("gate   :", verdict["gate_decision"])
display(pd.DataFrame([{"day": d, **r} for d, r in verdict["partitions"].items()]))
print(right_store.get_bytes(f"reports/{PAIR}_compare_row.csv").decode())
show_keys(right_store, "after the Right row check")

In [ ]:
# The manifest the Right side released (if the gate passed): windows skip every non-matching day
if right_store.exists(f"manifest/{PAIR}.json"):
    manifest_doc = right_store.get_json(f"manifest/{PAIR}.json")
    print("columns:", manifest_doc["columns"], "| sample:", manifest_doc["sample"], "| vintage:", manifest_doc["vintage"])
    display(pd.DataFrame(manifest_doc["windows"]))
    holes = [d for d, r in verdict["partitions"].items() if r["verdict"] != "match"]
    print("non-matching days:", holes, "| any of them in a window:",
          any(d in w["partitions"] for d in holes for w in manifest_doc["windows"]))
else:
    print("no manifest: the gate did not pass; Stage 2 below will find nothing to do")

## 3 · Left: pull windows, compute statistics, commit units

The Left side waits for `row_check/_CLOSED`, publishes fixed-N sample keys if the pair uses them, then for every window pulls the Left rows, samples, computes statistics and commits one unit per column (payload `units/...` first, then the `_INDEX/` pointer). When everything is indexed it writes `manifest/_CLOSED`.

In [ ]:
# Released manifests (returns at once: the row check is already closed)
manifests = wait_for_manifests(left_store, timeout=60, poll_interval=2)
total_rows = sum(w["n_left"] for m in manifests for w in m["windows"])
print(len(manifests), "manifest(s),", sum(m["expected_units"] for m in manifests), "units,", total_rows, "Left rows to pull")
if total_rows > MAX_ROWS:
    raise RuntimeError(f"{total_rows} Left rows > MAX_ROWS={MAX_ROWS}: narrow DAYS or raise MAX_ROWS")

# Fixed-N sampling only: the Left side chooses the keys once and shares them via manifest/<pair>.keys.json
for manifest in manifests:
    published_keys = publish_fixed_keys(left_store, manifest, WORKDIR / "left-keys")
    print(manifest["pair"], "fixed keys:", {k: len(v) for k, v in published_keys["windows"].items()} if published_keys else "n/a (not fixed-N)")

# The pipeline. wip is raised because the Right side is not draining concurrently in this notebook
runtime = LeftRuntime(WORKDIR / "left", left_store)
committed = ColCheckPipeline(left_store, pull_window=runtime.pull_window, compute_stats=runtime.compute_stats,
                             wip=10**9).run(manifests)
print("units committed:", committed)
print("manifest/_CLOSED:", left_store.get_json("manifest/_CLOSED") if left_store.exists("manifest/_CLOSED") else "not written (partial)")
show_keys(left_store, "after the Left col check")

In [ ]:
# One committed unit: the pointer in _INDEX/ and the statistics payload it points to
first_pointer_key = left_store.list_keys("_INDEX/")[0]
pointer = left_store.get_json(first_pointer_key)
print(first_pointer_key, "->", pointer)
display(pd.Series(left_store.get_json(pointer["payload_key"])))

## 4 · Right: drain and compare

`ColCheck` loads the manifests, pulls each Right window once (statistics computed locally from the warehouse rows), and compares every unit, writing `results/<unit>.json` (or `dead/<unit>.json` after `max_attempts` failures). It returns when `manifest/_CLOSED` is present and every unit is accounted for.

In [ ]:
# heartbeat_timeout is raised: in this notebook the Left side stopped beating when its cell finished
results = ColCheck(right_store, target_factory, poll_interval=1, idle_timeout=600, heartbeat_timeout=24 * 3600).run()
table = pd.DataFrame([{k: r.get(k) for k in ("pair", "vintage", "column", "right_col", "col_type", "n_total_left",
                                                 "n_total_right", "n_unique_left", "n_unique_right", "hash_sum_match", "verdict")}
                      for r in results])
display(table.sort_values(["vintage", "column"]) if len(table) else "no results")
dead = right_store.list_json("dead/")
print("dead units:", len(dead))
for key, failure in dead[:5]:
    print(" ", key, failure.get("error"))
show_keys(right_store, "after the Right col check")

## 5 · Report: build, publish, fetch back

The Right job builds the HTML / XLSX / CSV / `summary.json` set, publishes it to the report store (`history/<run_id>/` then `latest/pointer.json`), and the Left side fetches it with `fetch_report`. Here the report store is a sub-folder of the scratch prefix.

In [ ]:
# Build locally, publish to the scratch report store
files = build_report(right_store, WORKDIR / "report")
summary = json.loads(files["summary.json"].read_text())
print("status:", summary["status"], "| findings:", summary["findings"], "| units:", summary["units_completed"], "/", summary["units_expected"])
published = publish_report(open_store(REPORT_URI), summary["run_id"], files, summary)
print("published:", published)

# The Left side reads latest/pointer.json and downloads that run's files
fetched = fetch_report(LeftS3Store(REPORT_URI), WORKDIR / "fetched")
print("fetched run:", fetched["run_id"], "| same run:", fetched["run_id"] == RUN_ID)
for name, path in sorted(fetched["files"].items()):
    print(f"  {name:35} {path.stat().st_size:>9} bytes")
print("open in a browser:", (WORKDIR / "fetched" / "index.html").as_uri())

## 6 · The same flow in one call (dtrack_local)

`dtrack_local.run_all` runs preflight, Stage 1, Stage 2 and the report in a local run folder laid out like an S3 run, then backs the run folder up to S3. This is the prod-plan end-to-end check. Off by default (`RUN_LOCAL_E2E`); it pulls the same data again.

In [ ]:
if RUN_LOCAL_E2E:
    from dtrack_local import pipeline

    # Same pair and window; the Right side again through the warehouse; backup goes under the scratch prefix
    local_run = pipeline.open_run(
        run_config,
        run_id=f"{RUN_ID}-local",
        out=WORKDIR / "local-runs",
        target_factory=lambda spec: pipeline.WarehouseTarget(warehouse, spec, WORKDIR / "local-right"),
    )
    outcome = pipeline.run_all(local_run, gate=GATE, backup_s3=f"{SCRATCH}/local-backup")
    print("status:", outcome["status"], "| findings:", outcome["findings"])
    print("row check:", outcome["row_check"])
    print("col check:", outcome["col_check"])
    print("backup:", outcome["backup"])
    print("landing tables:", outcome["landing"])
    print("dead units:", len(local_run.store.list_keys("dead/")), "| report:", (local_run.folder / "report" / "index.html").exists())
else:
    print("skipped (set RUN_LOCAL_E2E = True)")

## 7 · Packaging: what ships where

Read-only checks of the installed packages: the Left package's dependencies, the import boundaries between `dtrack_left`, `dtrack_right` and `dtrack_local`, what the Right job installs on a Databricks runtime, and the `dtrack` entry point. Then the exact file list a Workspace upload of `src/dtrack_right` would create (`DRY_RUN`).

In [ ]:
import importlib.metadata as metadata
import tomllib

# The installed distribution: version, runtime dependencies, extras and the console script
print("dtrack", metadata.version("dtrack"))
for requirement in metadata.requires("dtrack") or []:
    print("  requires", requirement)
print("  console scripts:", [(e.name, e.value) for e in metadata.entry_points(group="console_scripts") if e.name == "dtrack"])

# The same from pyproject.toml: no databricks / pyathena / impyla in the base dependencies
project = tomllib.loads((REPO / "pyproject.toml").read_text())["project"]
base = project["dependencies"]
print("base deps:", base)
print("extras   :", project["optional-dependencies"])
print("databricks/athena/impyla in base deps:", [d for d in base if any(x in d for x in ("databricks", "pyathena", "impyla"))] or "none")

In [ ]:
import dtrack_local
import dtrack_right

# Import boundaries, scanned on the installed sources; every list should be empty
left_root = Path(dtrack_left.__file__).parent
right_root = Path(dtrack_right.__file__).parent
local_root = Path(dtrack_local.__file__).parent
pipeline_files = [*(left_root / "checks").glob("*.py"), *(left_root / "common").glob("*.py"), left_root / "cli/checks.py",
                  left_root / "backends/sources.py", left_root / "backends/store.py"]
helper = left_root / "backends/databricks.py"
scans = {
    "dtrack_right files that mention dtrack_left": [p for p in right_root.rglob("*.py") if "dtrack_left" in p.read_text()],
    "dtrack_left files with DATABRICKS_TOKEN": [p for p in left_root.rglob("*.py") if "DATABRICKS_TOKEN" in p.read_text()],
    "dtrack_left files importing databricks.sdk (besides the dbx helper)": [p for p in left_root.rglob("*.py") if p != helper and "databricks.sdk" in p.read_text()],
    "top-level databricks imports in the dbx helper": [l for l in helper.read_text().splitlines() if l.startswith(("import ", "from ")) and "databricks" in l],
    "Left pipeline modules importing databricks or the AWS helper": [p for p in pipeline_files if any(s in p.read_text() for s in ("databricks import", "import databricks", "import aws", "backends.aws"))],
    "dtrack_left / dtrack_right files importing dtrack_local": [p for side in (left_root, right_root) for p in side.rglob("*.py") if "import dtrack_local" in p.read_text() or "from dtrack_local" in p.read_text()],
}
for label, hits in scans.items():
    print(f"{label:70} {hits or 'none'}")
local_text = "\n".join(p.read_text() for p in local_root.glob("*.py"))
print("dtrack_local combines both sides:", "from dtrack_left" in local_text and "from dtrack_right" in local_text)

In [ ]:
import subprocess
import sys

# What the Right job installs on a Databricks runtime (the notebook's %pip cell)
print("dtrack_right/requirements.txt:", (right_root / "requirements.txt").read_text().split())

# `python -m dtrack` is the same CLI as the `dtrack` script
import dtrack
import dtrack_left.cli
print("dtrack.main is dtrack_left.cli.main:", dtrack.main is dtrack_left.cli.main)
print(subprocess.run([sys.executable, "-m", "dtrack", "--version"], capture_output=True, text=True, check=True).stdout.strip())

In [ ]:
import zipfile

# Optional: build the wheel into WORKDIR (never into the repo) and list what it bundles per package
if BUILD_WHEEL and shutil.which("uv"):
    dist = WORKDIR / "dist"
    subprocess.run(["uv", "build", "--wheel", "--out-dir", str(dist)], cwd=REPO, check=True)
    wheel = sorted(dist.glob("*.whl"))[-1]
    names = zipfile.ZipFile(wheel).namelist()
    per_package = {}
    for name in names:
        per_package.setdefault(name.split("/")[0], []).append(name)
    print(wheel.name)
    for package, items in sorted(per_package.items()):
        print(f"  {package:25} {len(items):3} files")
else:
    print("skipped (set BUILD_WHEEL = True; needs `uv` on PATH)")

In [ ]:
from dtrack_left.backends.databricks import upload_workspace

# The Workspace upload of the Right job folder: DRY_RUN lists the remote paths only (read-only: it asks who you are).
# The folder must be named dtrack_right: notebook.py puts its parent folder on sys.path.
plan = upload_workspace(workspace(DBX_PROFILE), REPO / "src" / "dtrack_right", WORKSPACE_TARGET,
                        overwrite=OVERWRITE, dry_run=DRY_RUN)
print({k: v for k, v in plan.items() if k != "paths"})
for path in plan["paths"]:
    print("  ", path)
if DRY_RUN:
    print("DRY_RUN: nothing was written to the Workspace (notebook.py would land as the notebook `notebook`)")

## Cleanup

Deletes everything under this run's scratch sub-folder (`SCRATCH`) and the local `WORKDIR`, unless `KEEP = True`.

In [ ]:
# Delete every object under SCRATCH with the same S3 client the stores use
if KEEP:
    print("kept", SCRATCH, "and", WORKDIR)
else:
    cleaner = LeftS3Store(SCRATCH)
    prefix = cleaner.location.prefix + "/"
    deleted = 0
    for page in cleaner.client.get_paginator("list_objects_v2").paginate(Bucket=cleaner.bucket, Prefix=prefix):
        batch = [{"Key": row["Key"]} for row in page.get("Contents") or []]
        if batch:
            cleaner.client.delete_objects(Bucket=cleaner.bucket, Delete={"Objects": batch})
            deleted += len(batch)
    print(f"deleted {deleted} object(s) under {SCRATCH}")
    shutil.rmtree(WORKDIR, ignore_errors=True)
    print("removed", WORKDIR)

## What to check

- Section 0: the Right store reads and lists what the Left store wrote.
- Section 1-2: `row_check/<pair>.left.json` carries no credentials; the gate passes and the manifest windows contain no non-matching day.
- Section 3: `units committed` = `expected_units`, `manifest/_CLOSED` written.
- Section 4: one result per unit, `dead units: 0`; `diff` verdicts are real findings to inspect.
- Section 5: `status DONE`, fetched run id = this run, `index.html` opens.
- Section 7: every boundary scan `none`, requirements are only `boto3` + `openpyxl`, the upload plan lists `dtrack_right/...` with `notebook` and no `.pyc` / ignored files.
- Cleanup deleted the scratch objects (unless `KEEP`).